In [1]:
!pip install -q accelerate jiwer soundfile librosa qwen_asr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 68.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 51.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 99.5 MB/s eta 0:00:00ta 0:00:01


In [2]:
!pip uninstall transformers -y
!pip install transformers==4.57.6

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)


In [8]:
import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

In [9]:
import os
import time
import torch
from jiwer import wer
from tqdm import tqdm
from qwen_asr import Qwen3ASRModel

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN VÀ MÔ HÌNH
# ==========================================
# Thay đổi ROOT_DIR thành đường dẫn thực tế của dataset vivos trên Kaggle
# Ví dụ: "/kaggle/input/vivos-dataset/vivos/test"
ROOT_DIR = "/kaggle/input/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr/vivos/test" 
PROMPTS_FILE = os.path.join(ROOT_DIR, "prompts.txt")
WAVES_DIR = os.path.join(ROOT_DIR, "waves")

MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
SAMPLING_RATE = 16000

# ==========================================
# CHUẨN BỊ DỮ LIỆU
# ==========================================
print("Đang đọc file prompts.txt và xây dựng đường dẫn...")
dataset = []

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if not line:
        continue
        
    # Tách ID và phần văn bản (transcript)
    parts = line.split(" ", 1)
    if len(parts) == 2:
        file_id, ground_truth = parts
        
        # Suy ra tên thư mục từ file_id (VD: VIVOSDEV02 từ VIVOSDEV02_R106)
        folder_name = file_id.split("_")[0]
        
        # Xây dựng đường dẫn tuyệt đối đến file wav
        wav_path = os.path.join(WAVES_DIR, folder_name, f"{file_id}.wav")
        
        if os.path.exists(wav_path):
            dataset.append({
                "wav_path": wav_path,
                "ground_truth": ground_truth
            })
        else:
            print(f"[Cảnh báo] Không tìm thấy file audio: {wav_path}")

print(f"Tổng số mẫu hợp lệ tìm thấy: {len(dataset)}")

# ==========================================
# KHỞI TẠO MÔ HÌNH
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang tải mô hình {MODEL_ID} lên {device}...")

# Khởi tạo theo thư viện qwen_asr
model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    max_inference_batch_size=32,
    max_new_tokens=256,
)

# ==========================================
# INFERENCE VÀ TÍNH TOÁN CHỈ SỐ
# ==========================================
predictions = []
references = []
total_inference_time = 0.0

print("Bắt đầu quá trình inference...")
with torch.no_grad():
    for item in tqdm(dataset, desc="Đang xử lý"):
        wav_path = item["wav_path"]
        ground_truth = item["ground_truth"]
        
        # Đo thời gian suy luận
        start_time = time.time()
        
        # Tuỳ thuộc vào API cụ thể của Qwen3ASRModel, hàm transcribe có thể khác một chút.
        # Ở đây giả định dùng method mặc định như .transcribe() hoặc model()
        # Nếu model nhận file path trực tiếp:
        pred_text = model.transcribe(wav_path, language="Vietnamese") 
        output_text = pred_text[0].text
        end_time = time.time()
        
        # Cộng dồn thời gian
        inference_time = end_time - start_time
        total_inference_time += inference_time
        
        # Lưu lại để tính WER (chuyển về chữ thường để so sánh công bằng)
        predictions.append(output_text.lower())
        references.append(ground_truth.lower())

# ==========================================
# TỔNG HỢP KẾT QUẢ
# ==========================================
num_samples = len(dataset)
avg_inference_time = total_inference_time / num_samples if num_samples > 0 else 0

# Tính Average Word Error Rate
# Lưu ý: Hàm wer của jiwer nhận tham số (reference, hypothesis)
avg_wer = wer(references, predictions)

print("\n" + "="*40)
print("KẾT QUẢ ĐÁNH GIÁ (TEST SET)")
print("="*40)
print(f"Tổng số mẫu đã xử lý: {num_samples}")
print(f"Average WER (Word Error Rate): {avg_wer * 100:.2f}%")
print(f"Average Inference Time per sample: {avg_inference_time:.4f} seconds")
print("="*40)

Đang đọc file prompts.txt và xây dựng đường dẫn...
Tổng số mẫu hợp lệ tìm thấy: 760
Đang tải mô hình Qwen/Qwen3-ASR-0.6B lên cuda...
Bắt đầu quá trình inference...


Đang xử lý:   9%|▉         | 68/760 [00:42<07:12,  1.60it/s]


KeyboardInterrupt: 

In [12]:
import os
import time
import torch
from jiwer import wer
from tqdm import tqdm
from qwen_asr import Qwen3ASRModel

# --- Cấu hình ---
DATASET_ROOT = "/kaggle/input/datasets/kynthesis/vivos-vietnamese-speech-corpus-for-asr/vivos"
MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
BATCH_SIZE = 32  # Điều chỉnh tùy theo VRAM (thường 16-32 cho 0.6B model)

def load_vivos_data(split_dir):
    prompts_file = os.path.join(split_dir, "prompts.txt")
    waves_dir = os.path.join(split_dir, "waves")
    data = []
    
    if not os.path.exists(prompts_file):
        return []

    with open(prompts_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(" ", 1)
            if len(parts) == 2:
                file_id, ground_truth = parts
                folder_name = file_id.split("_")[0]
                wav_path = os.path.join(waves_dir, folder_name, f"{file_id}.wav")
                if os.path.exists(wav_path):
                    data.append({"path": wav_path, "gt": ground_truth.lower()})
    return data

def evaluate_split(model, dataset, split_name="Test"):
    print(f"\n--- Đang đánh giá tập: {split_name} ({len(dataset)} mẫu) ---")
    predictions = []
    references = [item['gt'] for item in dataset]
    paths = [item['path'] for item in dataset]
    
    total_time = 0.0
    
    # Chia batch để inference
    for i in tqdm(range(0, len(paths), BATCH_SIZE), desc=f"Inference {split_name}"):
        batch_paths = paths[i : i + BATCH_SIZE]
        
        start_time = time.time()
        # Giả định model.transcribe hỗ trợ list paths và trả về list object có thuộc tính .text
        batch_results = model.transcribe(batch_paths, language="Vietnamese")
        end_time = time.time()
        
        total_time += (end_time - start_time)
        
        # Trích xuất text từ kết quả batch
        for res in batch_results:
            predictions.append(res.text.lower())

    avg_wer = wer(references, predictions)
    avg_time = total_time / len(dataset)
    
    return avg_wer, avg_time

# --- Main Execution ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto", # auto sẽ tự tối ưu hóa pipeline
)

splits = {
    "Train": os.path.join(DATASET_ROOT, "train"),
    "Test": os.path.join(DATASET_ROOT, "test")
}

results = {}

for name, path in splits.items():
    dataset = load_vivos_data(path)
    if dataset:
        wer_val, time_val = evaluate_split(model, dataset, name)
        results[name] = {"WER": wer_val, "Time": time_val}

# --- In kết quả tổng hợp ---
print("\n" + "="*50)
print(f"{'Split':<10} | {'WER (%)':<12} | {'Avg Time (s)':<15}")
print("-" * 50)
for split, metrics in results.items():
    print(f"{split:<10} | {metrics['WER']*100:>10.2f}% | {metrics['Time']:>13.4f}s")
print("="*50)


--- Đang đánh giá tập: Train (11660 mẫu) ---


Inference Train: 100%|██████████| 365/365 [30:45<00:00,  5.06s/it]



--- Đang đánh giá tập: Test (760 mẫu) ---


Inference Test: 100%|██████████| 24/24 [01:38<00:00,  4.11s/it]


Split      | WER (%)      | Avg Time (s)   
--------------------------------------------------
Train      |       8.12% |        0.1583s
Test       |      10.63% |        0.1299s
